<!-- @@FENCINGLY_DURACION_COMPLETA:protocolo -->
# 00 - REGLA DE PROTOCOLO. Leer antes de ejecutar y antes de interpretar

**El numero que salga de esta corrida es una LINEA BASE NUEVA. No se puede comparar con el 73,88 %,
ni con el 70,04 %, ni con ninguna cifra anterior del proyecto.**

Con duracion completa cambian `pre_s_eff`, `n_frames_window`, cada rasgo derivado de la ventana y la
huella del productor. Es otra cadena de medida. Restar una cifra de un protocolo a una cifra de otro
protocolo es un **error de protocolo**, no una comparacion conservadora. El informe del proyecto ya
lleva esa disciplina escrita ("nowhere in this report is a figure from one protocol subtracted from a
figure of another") y aqui aplica igual.

Lo que **si** se puede comparar: duracion-completa contra duracion-completa, con y sin el cambio que
se este probando, **pareado por clip**.

## El veredicto se reporta en dos poblaciones, siempre

El atajo "gana el que enciende segundo" acierta **63,51 %** por si solo, y el modelo lo predice en el
**72,71 %** de los clips: lo sobreusa. Eso parte el corpus:

| poblacion | n | acierto |
|---|---:|---:|
| gano el que encendio **segundo** | 1.671 | **83,00 %** |
| gano el que encendio **primero** | 960 | **45,21 %** |
| brecha | | **+37,80 pp** |

* llevar los 960 de 45,21 % a **60 %** vale **+5,40 pp** globales; a **75 %**, **+10,95 pp**
* degradar 5 pp los 1.671 cuesta **-3,18 pp**

**El veredicto es la media global pareada.** El subgrupo describe donde cayo la ganancia; nunca es la
poblacion de evaluacion, nunca un objetivo de optimizacion y nunca una puerta de enrutado. Evaluar
dentro de un subgrupo definido por el resultado hace que la base caiga bajo el azar: alli un bloque
sin informacion compro **+3,08 pp**.

## Limitacion honesta, y va en el informe

Con la potencia de este corpus **un efecto real de +1,6 pp es indetectable**, y hacen falta **12 a 20
campeonatos mas** para tener el 80 % de potencia sobre ese tamano de efecto. **Una mejora real puede
no ser demostrable aunque exista.** Este cuaderno no promete lo que no puede medir.

## Varas de medir, para no creerse un delta

* Cambiar **solo** el backend de atencion, misma receta, misma semilla y mismo interprete, da
  **+0,36 pp con 327 de 3.047 decisiones distintas**. Cualquier mejora de red por debajo de eso es
  indistinguible de recompilar. (Por eso la celda de preflight fuerza el backend `math`.)
* El nulo se elige **por el tipo de rasgo**. El desplazamiento circular **preserva** los rasgos de
  nivel (una columna quedo identica al 100 %), **infla** los de derivada (hasta 7,79x) y es **inerte**
  sobre representaciones agrupadas (coseno 0,9980). Para nivel, nulo de **signo de lado aleatorio**.
* **Dos convenios de promediado de semillas no son la misma cantidad.** El banco de ablaciones
  promedia el *acierto por clip*; el artefacto promedia las *probabilidades* y umbraliza una vez. Un
  efecto de +1,03 con el primero fue **+0,80 con intervalo cruzando el cero** con el segundo. **Usa
  el del artefacto.**


# 01 · Datos y evidencia — primer paso efectivo

**Flujo recomendado: 01 → 02 → importar el workspace final en la web local.** El notebook 03 es opcional y abre exactamente
el mismo frontend React de la entrega; no es una tercera aplicación ni un Gradio que haya que mantener aparte.

Este notebook selecciona metadata, registra particiones globales, reutiliza MP4 locales o descarga sólo si lo autorizas,
extrae poses/luces/audio, conserva QC y movimientos para la web, construye un Store derivado y exporta un workspace sin
vídeos originales. Las celdas de ejecución son deliberadamente opt-in: por defecto no descargan, no cargan YOLO y no
procesan una colección completa al pulsar «Ejecutar todo». Primero configura y revisa; después activa `RUN_DATA=True`.

### Los números de partida, sin inflarlos

| Capa auditada | Cantidad real | Lo que NO significa |
|---|---:|---|
| JSON de fuente | 140.739 registros | No son 140.739 vídeos locales ni ejemplos entrenables |
| Vídeos en la colección local revisada | 1.100 MP4 / 1.097 IDs | Los duplicados o reencodificaciones no crean unidades independientes |
| NPZ v2 históricos | 962 archivos / 960 IDs | No tienen todos los marcadores temporales actuales; no se promocionan por su existencia |
| NPZ actuales de la demo anterior | 6 | Prueba de ingeniería, no dataset/precisión de producción |
| Objetivo futuro de expansión | 5.000 clips, si se obtienen y validan | No es un censo local ya disponible |

La fuente competitiva auditada llega a junio de 2023. Consultarla o copiarla en 2026 no amplía sus fechas deportivas.
Los 81 pseudo-gold históricos son etiquetas LLM, no arbitraje humano independiente. No se usan para certificar detectores.

El catálogo maestro de la web puede reunir 16 datasets de análisis de esta entrega. **Ese inventario no es un manifiesto
de entrenamiento**: incluye procedencias/versiones/posibles solapamientos distintos. No se concatena automáticamente para
entrenar. Cada selección exige registro GLOBAL, identidades, etiquetas, NPZ actuales y checks de particiones propios.


## Abrir el mismo release privado: ZIP subido, carpeta ya extraída o clon fijado por commit

Necesitas dos cosas diferentes: **el ZIP de la entrega FENCINGLY** (código de confianza) y, a partir del segundo notebook,
**el ZIP de workspace** (sólo datos/modelo). No son intercambiables. No subas un workspace en el lugar del release.
El código de esta entrega no tiene por qué existir en ningún repositorio remoto. No hay una descarga de GitHub por defecto.

La entrega recomendada tiene `FENCINGLY/webapp/fencingly_v2/` y `FENCINGLY/webapp/webapp_react/frontend/dist/`.
Los notebooks viven en `FENCINGLY/colab/notebooks/` y el estudio en `FENCINGLY/study/`. Después del bootstrap, **ROOT apunta
al contenido de `webapp/`**, no a su carpeta padre ni al workspace de datos.

**En Colab:** abre este `.ipynb` con Archivo → Subir notebook. Si quieres acelerar la extracción/entrenamiento, selecciona
Entorno de ejecución → Cambiar tipo de entorno → GPU. Colab decide qué GPU hay; una T4 no está garantizada. Sube el ZIP del
release cuando se solicite, o escribe su ruta en `RELEASE_ZIP` si está en Drive ya montado. Para un fichero grande, monta
Drive primero en una celda propia con `google.colab.drive.mount('/content/drive')` y usa su ruta, evitando cargarlo entero
con el navegador. Si conservas el código extraído en este mismo runtime, proporciona `RELEASE_ROOT` y no vuelvas a extraer.

**En un PC/Jupyter:** descomprime la entrega de confianza y fija `RELEASE_ROOT` a su carpeta `webapp` (por ejemplo
`C:/mis_datos/FENCINGLY/webapp`). No uses una ruta de esta auditoría en otro ordenador. Todos los paths posteriores se
calculan desde ese ROOT y desde tus carpetas explícitas.

La celda mínima de bootstrap sólo lee del ZIP un helper **cuyo SHA-256 está fijado en este notebook**. No ejecuta un helper
arbitrario procedente de un workspace. Ese helper valida las rutas de todo el release, tamaños, expansión, colisiones,
enlaces y presencia del servidor/React `dist` antes de extraer a una carpeta nueva. No usa `extractall`. El estudio y los
notebooks del ZIP completo no se duplican dentro de `webapp/`. Si tienes el SHA-256 publicado **junto a tu entrega privada**,
fíjalo en `EXPECTED_RELEASE_SHA256`: integridad sin una referencia de confianza no equivale a autenticidad del autor.
No cambies el hash del helper para hacer pasar un ZIP desconocido. Si actualizas el código, pide el notebook del mismo release.

### Tercera vía opcional: clonar un repositorio con el commit fijado

Si prefieres no volver a subir el ZIP en cada sesión, fija `FENCINGLY_RELEASE_GIT` a la URL HTTPS del repositorio,
`FENCINGLY_RELEASE_GIT_REF` a su rama o etiqueta y `FENCINGLY_RELEASE_COMMIT` al commit completo de 40 hex que hayas
revisado. Por defecto esas tres variables están vacías y **no se clona nada**: la vía por defecto sigue siendo el ZIP
privado subido a mano, y el flujo sin red externa no cambia. Publicar el código sigue siendo una decisión tuya, no un
requisito de estos notebooks.

El clon es superficial (`--depth 1`) sobre una carpeta nueva y **no relaja la integridad**. Una rama es móvil, así que
el pin de commit es **obligatorio** al clonar: si el `HEAD` clonado no es exactamente ese commit, la celda se detiene.
Después se comprueba además el SHA-256 del helper `fencingly_v2/artifact_bundle.py` contra `HELPER_SHA256`, igual que
en la vía del ZIP. Un commit fijado prueba qué árbol se ejecuta, no quién lo escribió: revisa el repositorio antes de
fijarlo y no actualices el pin sin leer el diff. No se hace `push`, no se publican datos y no se sube nada al clonar.

El repositorio debe exponer el `webapp` en **una sola** de estas tres formas, las mismas que acepta el ZIP:
`FENCINGLY/webapp/`, `webapp/` o la propia raíz. Dentro tienen que estar `fencingly_v2/`, `webapp_react/server.py` y
`webapp_react/frontend/dist/index.html` ya compilado; sin ese `dist` la celda se detiene y no se arranca otro frontend.

La procedencia elegida queda en `RELEASE_PROVENANCE` (origen, URL, ref, commit, hash del helper y si el pin coincidió),
se imprime y viaja con el workspace exportado.


In [ ]:
import os, sys, hashlib, shutil, subprocess, zipfile, types
from pathlib import Path
from datetime import datetime, timezone

RELEASE_ROOT = os.environ.get("FENCINGLY_RELEASE_ROOT", "")
RELEASE_ZIP = os.environ.get("FENCINGLY_RELEASE_ZIP", "")
RELEASE_GIT_URL = os.environ.get("FENCINGLY_RELEASE_GIT", "")
RELEASE_GIT_REF = os.environ.get("FENCINGLY_RELEASE_GIT_REF", "main")
EXPECTED_RELEASE_COMMIT = os.environ.get("FENCINGLY_RELEASE_COMMIT", "")
EXPECTED_RELEASE_SHA256 = os.environ.get("FENCINGLY_RELEASE_SHA256", "")
IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules)
HELPER_SHA256 = "ee0e2a0c5cebf80c41c9b5e15c43c48801173780682553e7f24bd9bbb58848e3"
RELEASE_LAYOUTS = ("FENCINGLY/webapp", "webapp", "")

if RELEASE_ROOT:
    ROOT = Path(RELEASE_ROOT).expanduser().resolve()
    RELEASE_PROVENANCE = {"source": "release_root", "location": str(ROOT), "helper_sha256": HELPER_SHA256,
                          "authenticity": "user_must_trust_release_source"}
elif RELEASE_GIT_URL:
    if not RELEASE_GIT_URL.startswith("https://") or any(character.isspace() for character in RELEASE_GIT_URL):
        raise ValueError("Sólo se clona un remoto HTTPS explícito; no se aceptan otros transportes")
    if not RELEASE_GIT_REF or RELEASE_GIT_REF.startswith("-") or any(character.isspace() for character in RELEASE_GIT_REF):
        raise ValueError("Indica una rama o etiqueta explícita en FENCINGLY_RELEASE_GIT_REF")
    if len(EXPECTED_RELEASE_COMMIT) != 40 or any(character not in "0123456789abcdef" for character in EXPECTED_RELEASE_COMMIT.lower()):
        raise ValueError("Clonar exige FENCINGLY_RELEASE_COMMIT con el commit completo de 40 hex que hayas revisado: una rama es móvil y no acredita integridad")
    if shutil.which("git") is None:
        raise ValueError("Este runtime no trae git; usa RELEASE_ZIP o RELEASE_ROOT")
    parent = Path("/content" if IN_COLAB else Path.cwd())
    clone_root = parent / ("fencingly_clone_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))
    if clone_root.exists():
        raise ValueError("El destino del clon ya existe; no se reutiliza ni se sobrescribe una carpeta previa")
    cloned = subprocess.run(["git", "clone", "--depth", "1", "--branch", RELEASE_GIT_REF, "--", RELEASE_GIT_URL, str(clone_root)],
                            capture_output=True, text=True)
    if cloned.returncode != 0 or not (clone_root / ".git").is_dir():
        raise ValueError("El clon superficial ha fallado: " + " ".join((cloned.stderr or "").split())[:300])
    resolved = subprocess.run(["git", "-C", str(clone_root), "rev-parse", "HEAD"], capture_output=True, text=True)
    cloned_commit = (resolved.stdout or "").strip().lower()
    if resolved.returncode != 0 or len(cloned_commit) != 40:
        raise ValueError("No se ha podido leer el commit clonado; no se ejecuta un árbol sin identificar")
    if cloned_commit != EXPECTED_RELEASE_COMMIT.lower():
        raise ValueError("El commit clonado no coincide con FENCINGLY_RELEASE_COMMIT: revisa el cambio y actualiza el pin; no se ejecuta un árbol no fijado")
    layouts = [clone_root.joinpath(*[part for part in layout.split("/") if part]) for layout in RELEASE_LAYOUTS]
    layouts = [candidate for candidate in layouts if (candidate / "fencingly_v2" / "artifact_bundle.py").is_file()]
    if len(layouts) != 1:
        raise ValueError("El repositorio debe exponer exactamente un webapp con fencingly_v2/artifact_bundle.py en FENCINGLY/webapp, en webapp o en la raíz")
    ROOT = layouts[0].resolve()
    cloned_helper = (ROOT / "fencingly_v2" / "artifact_bundle.py").read_bytes()
    if hashlib.sha256(cloned_helper).hexdigest() != HELPER_SHA256:
        raise ValueError("El helper clonado no coincide con este notebook: usa el commit del mismo release; no omitas el control")
    RELEASE_PROVENANCE = {"source": "git_clone", "url": RELEASE_GIT_URL, "ref": RELEASE_GIT_REF, "commit": cloned_commit,
                          "helper_sha256": HELPER_SHA256, "authenticity": "pinned_commit_matched"}
    print("Release clonado y fijado:", RELEASE_PROVENANCE)
else:
    if not RELEASE_ZIP:
        if not IN_COLAB:
            raise ValueError("Fija RELEASE_ROOT al webapp del release, RELEASE_ZIP al ZIP privado o RELEASE_GIT al repositorio con su commit fijado")
        from google.colab import files
        uploaded_release = files.upload()
        candidates = [name for name in uploaded_release if name.lower().endswith(".zip")]
        if len(candidates) != 1:
            raise ValueError("Sube exactamente un ZIP de RELEASE, no el workspace")
        RELEASE_ZIP = candidates[0]
        del uploaded_release
    if Path(RELEASE_ZIP).stat().st_size > 4 * 1024 ** 3:
        raise ValueError("Release mayor que el presupuesto de 4 GiB")
    with Path(RELEASE_ZIP).open("rb") as release_stream:
        footer = zipfile._EndRecData(release_stream)
        if footer is None or footer[zipfile._ECD_ENTRIES_TOTAL] > 30000 or footer[zipfile._ECD_SIZE] > 8 * 1024 ** 2:
            raise ValueError("Metadatos del ZIP de release excesivos o inválidos antes de cargar su directorio")
    with zipfile.ZipFile(RELEASE_ZIP) as archive:
        if len(archive.infolist()) > 30000:
            raise ValueError("Release con demasiadas entradas")
        approved_names = {prefix + "fencingly_v2/artifact_bundle.py" for prefix in ("FENCINGLY/webapp/", "webapp/", "")}
        helpers = [entry for entry in archive.infolist() if entry.filename in approved_names]
        if len(helpers) != 1 or helpers[0].file_size > 150000 or helpers[0].flag_bits & 1:
            raise ValueError("Helper del release ausente, ambiguo o excesivo")
        helper_bytes = archive.read(helpers[0])
    if hashlib.sha256(helper_bytes).hexdigest() != HELPER_SHA256:
        raise ValueError("El helper no coincide con este notebook: usa el release correspondiente; no omitas el control")
    bootstrap = types.ModuleType("_fencingly_verified_release_bootstrap")
    sys.modules[bootstrap.__name__] = bootstrap
    exec(compile(helper_bytes, "verified_artifact_bundle.py", "exec"), bootstrap.__dict__)
    parent = Path("/content" if IN_COLAB else Path.cwd())
    destination = parent / ("fencingly_release_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))
    extracted = bootstrap.extract_release_to_new_directory(RELEASE_ZIP, destination,
                    expected_sha256=EXPECTED_RELEASE_SHA256 or None)
    ROOT = Path(extracted["root"])
    RELEASE_PROVENANCE = {"source": "release_zip", "sha256": extracted["sha256"], "webapp_prefix": extracted["webapp_prefix"],
                          "helper_sha256": HELPER_SHA256, "authenticity": extracted["authenticity"]}
    print("Release validado:", extracted)

required = ("fencingly_v2/colab_workflow.py", "fencingly_v2/training_recipe.py", "fencingly_v2/epoch_checkpoint.py",
            "fencingly_v2/measurement_artifacts.py", "webapp_react/server.py", "webapp_react/frontend/dist/index.html")
if not all((ROOT / relative).is_file() for relative in required):
    raise ValueError("ROOT no contiene el webapp completo del mismo release, incluido React dist")
if "fencingly_v2" in sys.modules and Path(sys.modules["fencingly_v2"].__file__).resolve().parent != ROOT / "fencingly_v2":
    raise RuntimeError("Otro release ya está importado: reinicia el kernel; no mezcles bindings de fuentes distintas")
sys.path.insert(0, str(ROOT))
os.environ["FENCINGLY_RELEASE_ROOT"] = str(ROOT)
from fencingly_v2 import colab_workflow as CW, artifact_bundle as AB
print("ROOT del código compartido:", ROOT)


## Dependencias: comprobar antes de instalar

La entrega no presupone que Colab traiga las versiones correctas. El informe muestra Python y módulos realmente
instalados. La instalación está **desactivada**. Si falta algo, activa `INSTALL_MISSING=True` de forma consciente;
el helper sólo solicita los módulos de esa fase. Nunca instala automáticamente torch/CUDA cuando torch falta: elige
el runtime y la distribución oficial adecuados, reinicia si el instalador lo pide y repite la comprobación. No copies un
entorno virtual de Windows a Colab. No se ha instalado aquí un stack GPU para aparentar una prueba en CUDA.

Un import correcto no verifica la compatibilidad de un Store o modelo: esos artefactos conservan sus propios contratos.
La parte web puede necesitar dependencias opcionales adicionales para informes/PDF; usa las requirements de la misma
entrega y los mensajes del servidor, no una lista de paquetes de otro notebook histórico.


In [ ]:
INSTALL_MISSING = False
DATA_DEPENDENCIES = CW.dependency_report("data")
print(DATA_DEPENDENCIES)
if INSTALL_MISSING:
    DATA_DEPENDENCIES = CW.install_missing("data", approved=True)


## Dónde viven los datos y cómo sobrevivir a una desconexión

`WORK` es una carpeta **nueva de trabajo**, distinta de ROOT y de tus originales. En Colab se procesa sobre disco local:
Drive es lento con miles de ficheros pequeños. Se copian snapshots ZIP verificados a Drive y el registro GLOBAL se mantiene
allí directamente. En local puedes elegir un disco propio. No reutilices la carpeta histórica de otra versión como destino.

`USE_DRIVE=True` en Colab pide tu autorización para montar Drive. Si no quieres hacerlo, fija False y descarga el ZIP al
terminar cada fase; perder el runtime antes de esa descarga pierde lo que sólo estuviese en `/content`. Un path que parece
Drive pero no está montado **no** es persistencia. La copia comprueba el hash del fichero visible; no demuestra que Google
haya terminado de sincronizarlo remotamente. No cierres la sesión inmediatamente después de copiar.

`files.download` inicia una transferencia hacia el navegador, no confirma que haya terminado. Puede empezar al finalizar
la celda y mantiene el fichero en RAM del navegador. El helper recomienda Drive y rechaza esa vía para ZIPs mayores de
512 MiB. No hay auto-clicks para evitar el timeout, túneles públicos ni tareas ocultas.


In [ ]:
from fencingly_v2 import colab_utils as CU
USE_DRIVE = IN_COLAB
DRIVE_DIR = Path("/content/drive/MyDrive/FENCINGLY") if USE_DRIVE else None
if USE_DRIVE:
    if not IN_COLAB:
        raise ValueError("En local usa una carpeta explícita; este montaje es sólo de Colab")
    CU.mount_drive()
    if not CU.drive_mounted():
        raise RuntimeError("Drive no está montado; no se simula una copia persistente")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RUN_PARENT = Path(os.environ.get("FENCINGLY_RUN_PARENT", "/content/fencingly_runs" if IN_COLAB else str(ROOT.parent / "colab_runs")))
RUN_PARENT.mkdir(parents=True, exist_ok=True)
WORK = Path(os.environ.get("FENCINGLY_WORKSPACE", str(RUN_PARENT / ("data_" + CW.stamp()))))
WORKSPACE_TO_RESUME = os.environ.get("FENCINGLY_WORKSPACE_ZIP", "")
if WORKSPACE_TO_RESUME:
    resumed = AB.restore_to_new_directory(WORKSPACE_TO_RESUME, WORK)
    print("Restaurado en carpeta nueva; snapshot NO adoptado como autoridad global:", resumed["destination"])
else:
    WORK.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = RUN_PARENT / "exports"
EXPORT_DIR.mkdir(exist_ok=True)
print("WORK:", WORK, "| Drive:", DRIVE_DIR)


## Configuración explícita: fuente, arma, originales y registro GLOBAL

1. **SOURCE_JSON:** JSON local que enumera los clips. Puede estar en Drive. No se vuelve a descargar automáticamente.
2. **WEAPON:** una de `foil`, `epee`, `sabre`. Para preparar otra arma crea otro WORK, pero utiliza el **mismo archivo de
registro GLOBAL**. La receta supervisada del notebook 02 sigue restringida a florete post-tocado. Las dobles de espada no
se convierten en prioridad binaria de florete.
3. **LOCAL_CLIPS_DIRS / LOCAL_PATHS:** carpetas tuyas o diccionario `clip_id → MP4`. Leerlos no los mueve ni los borra.
Si hay dos rutas para el mismo ID, el helper exige que resuelvas la ambigüedad; no selecciona una copia por mtime ni cuenta
ambas como datos nuevos. No hace falta subir todos los vídeos para usar un workspace NPZ ya validado.
4. **GLOBAL_REGISTRY:** ruta explícita, persistente y fuera de WORK/Store. Ejemplo Colab:
`/content/drive/MyDrive/FENCINGLY/tournament_splits_GLOBAL.json`. Local: una carpeta `state/` común a todas tus armas/runs.
Los registros históricos deben conservarse. `CREATE_NEW_GLOBAL_REGISTRY=True` sólo es apropiado para un proyecto realmente
nuevo sin asignaciones anteriores; no reconstituye mágicamente un test no visto. `PREVIOUS_MANIFEST` permite registrar
historia explícita. Si hay cruces conocidos por torneo/asalto/clip/vídeo, se detiene: no se arregla renombrando carpetas.
5. **N_CLIPS / CLIP_IDS:** límite positivo o lista explícita. Sin lista se usa selección determinista alternando torneos,
no «los primeros miles» de un atleta. Los filtros tabulares de coherencia no prueban la etiqueta ni la calidad anatómica.
En CPU la política inicial permite como máximo seis clips; empieza con cuatro y comprueba datos/tiempos antes de crecer.

La asignación de particiones se realiza antes del Store. El manifiesto del WORK queda ligado al hash de su fuente; si
cambia el JSON, crea otra versión de WORK. Nunca se reescribe `Store/meta.csv` para cambiar sus splits. La selección del
notebook 02 es una vista externa validada, no un cambio oculto de digest.

### Una sola pregunta: qué porcentaje del florete quieres analizar

La celda siguiente tiene **una variable visible**, `PERCENT_OF_FOIL_CLIPS`. Todo lo demás se deduce del runtime que
Colab te haya dado. Empieza con `1.0` para un ensayo rápido y sube a `100.0` cuando quieras el corpus completo.

Lo que antes había que decidir a mano y ahora se deriva solo: el número de clips, el perfil de pose
(`balanced960` con GPU, `fast640` sin ella), los hilos de Torch, el tamaño de la cola de descarga y de la
ventana decodificada, cada cuánto se copia un snapshot, el presupuesto de descarga y el borrado de los MP4.
También se descargan solos el catálogo público y los pesos YOLO-pose si no los tienes ya, diciéndote de dónde.
Las variables de entorno `FENCINGLY_*` siguen mandando sobre cualquier valor derivado.

**El muestreo es proporcional por torneo, no «los primeros N».** Cada torneo aporta a la muestra la misma
fracción que tiene en el corpus (reparto por mayor resto), y dentro de cada torneo se alternan los asaltos
ordenados por SHA-256 de «semilla|clave». Como las particiones están congeladas por torneo, esto hace que la
proporción train/val/test de un 1% sea la misma que la del 100%: un ensayo pequeño es un ensayo representativo,
no una esquina del corpus. La semilla es fija, así que la misma peticion devuelve siempre los mismos clips.

**Antes de trabajar horas, el notebook juzga el runtime en voz alta.** `runtime_verdict` estima el coste con la
selección pedida y se detiene con un mensaje claro si no hay GPU o falta RAM, en vez de fallar clip a clip al
final. Los clips que no puedan procesarse se cuentan por motivo y se imprimen: nunca desaparecen en silencio.


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:config
import sys as _sys
from pathlib import Path as _P
for _c in (_P.cwd(), _P.cwd()/"paquete", _P("/content"), _P("/content/paquete")):
    if (_c/"preflight.py").is_file() and str(_c) not in _sys.path:
        _sys.path.insert(0, str(_c))
import preflight as PF

# ALMACENAR Y ENTRENAR SON DOS DECISIONES DISTINTAS. Estos son los mandos, y son cuatro.
#
# 1) Que se GUARDA en el NPZ. Cuesta GPU y High-RAM, y compra no volver a descargar 93 GB.
#      "desplegada" 4.0/0.3 | "amplia" 6.0/1.0 | "completa" 60/60 (el clip entero)
VENTANA_ALMACEN = os.environ.get("FENCINGLY_VENTANA_ALMACEN", "completa")
#
# 2) La RODAJA sobre la que se mide y se entrena, derivada del NPZ guardado. No cuesta nada mas.
#      "historica" 4.0/0.3 (130 cuadros, T_max 135)
#      "recomendada" 4.0/0.6 (139 cuadros, T_max 140)  <- cierra la unica brecha MEDIDA
#      "ancha" 6.0/1.0 (211 cuadros, T_max 211)        <- cambia lo que ve la red; NO medido
RODAJA_ENTRENAMIENTO = os.environ.get("FENCINGLY_RODAJA", "recomendada")
#
# 3) T_max del Store. Tiene que admitir la rodaja o la ventana de guardia se pierde en silencio.
T_MAX = int(os.environ.get("FENCINGLY_T_MAX", "140"))
#
# 4) El reloj. Apagarlo lleva la ventana de guardia de 72,78 % a 91,42 % A COSTE CERO: es la
#    ganancia mas barata de todo el barrido. AVISO: hay trabajo en curso para CONECTAR la tabla del
#    reloj, asi que las dos decisiones INTERACTUAN y hay que medirlas juntas en un 2x2, no una
#    encima de la otra. Si el otro cambio ya esta dentro, declara las dos en el informe.
USAR_RELOJ = os.environ.get("FENCINGLY_USAR_RELOJ", "0") == "1"

PRE_S, POST_S, _use_clock_por_defecto = PF.VENTANAS[VENTANA_ALMACEN]
USE_CLOCK = USAR_RELOJ
RODAJA_PRE_S, RODAJA_POST_S = PF.RODAJAS[RODAJA_ENTRENAMIENTO]
print(f"ALMACEN     {VENTANA_ALMACEN!r}: pre_s={PRE_S} post_s={POST_S} use_clock={USE_CLOCK}")
print(f"ENTRENA     {RODAJA_ENTRENAMIENTO!r}: pre_s={RODAJA_PRE_S} post_s={RODAJA_POST_S} T_max={T_MAX}")

PERCENT_OF_FOIL_CLIPS = 1.0

WEAPON = os.environ.get("FENCINGLY_WEAPON", "foil")
EXPLICIT_N_CLIPS = int(os.environ.get("FENCINGLY_N_CLIPS", "0")) or None
PERCENT = None if EXPLICIT_N_CLIPS else float(os.environ.get("FENCINGLY_PERCENT", PERCENT_OF_FOIL_CLIPS))
RUN_DATA = os.environ.get("FENCINGLY_RUN_DATA", "1" if IN_COLAB else "0") == "1"
SOURCE_JSON = os.environ.get("FENCINGLY_SOURCE_JSON", "")
GLOBAL_REGISTRY = os.environ.get("FENCINGLY_SPLIT_REGISTRY", "")
POSE_WEIGHTS = os.environ.get("FENCINGLY_POSE_WEIGHTS", "")
CLIP_IDS = None
LOCAL_CLIPS_DIRS = AB.strict_json(os.environ.get("FENCINGLY_LOCAL_CLIPS_DIRS", "[]"))
LOCAL_PATHS = {}
PREVIOUS_MANIFEST = None
FIE_CACHE_SOURCE = os.environ.get("FENCINGLY_FIE_CACHE_SOURCE", "")
RUN_FIE_PREFETCH = False
FIE_ONLINE = False
FIE_EVENTS = []
VERIFIED_FIE_IDS = {}
BUILD_STORE = True
DOWNLOAD_FINAL_IN_BROWSER = IN_COLAB
SOURCE_URL = "https://clips.fencingdatabase.com/all_gfy_info.json"
POSE_WEIGHTS_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11m-pose.pt"

SELECTED = None
SELECTION_INFO = None
ENVIRONMENT = None
PLAN = None
VERDICT = None
if RUN_DATA:
    if not SOURCE_JSON:
        if not IN_COLAB:
            raise ValueError("Fija FENCINGLY_SOURCE_JSON al catalogo de clips: la descarga automatica solo actua en Colab")
        SOURCE_JSON = str(RUN_PARENT / "all_gfy_info.json")
        if not Path(SOURCE_JSON).is_file():
            print("Descargando el catalogo publico de clips desde", SOURCE_URL, flush=True)
            CW.download_url(SOURCE_URL, SOURCE_JSON)
        print("Catalogo:", SOURCE_JSON, Path(SOURCE_JSON).stat().st_size, "bytes")
    if not GLOBAL_REGISTRY:
        GLOBAL_REGISTRY = str((DRIVE_DIR or RUN_PARENT) / "tournament_splits_GLOBAL.json")
        print("Registro GLOBAL de particiones:", GLOBAL_REGISTRY)
    CREATE_NEW_GLOBAL_REGISTRY = not Path(GLOBAL_REGISTRY).is_file()
    if CREATE_NEW_GLOBAL_REGISTRY:
        print("No existe todavia: se creara uno nuevo. Consérvalo en Drive; perderlo rompe la separacion train/test")
    ENVIRONMENT = CW.environment_report(WORK, stage="data")
    if ENVIRONMENT["dependencies"]["missing"]:
        print("Instalando lo que falta para la fase de datos:", ENVIRONMENT["dependencies"]["missing"], flush=True)
        CW.install_missing("data", approved=True)
        ENVIRONMENT = CW.environment_report(WORK, stage="data")
    if ENVIRONMENT["dependencies"]["missing"] or not ENVIRONMENT["ffmpeg"]["available"]:
        raise RuntimeError(f"Dependencias/ffmpeg incompletos: {ENVIRONMENT['dependencies']['missing']}")
    if not LOCAL_PATHS:
        LOCAL_PATHS = CW.local_video_map(LOCAL_CLIPS_DIRS, recursive=True)
    POSE_PROFILE = "balanced960" if ENVIRONMENT["resources"]["gpu"].get("available") else "fast640"
    TORCH_THREADS = 1
    POPULATION = CW.eligible_population(SOURCE_JSON, weapon=WEAPON)
    TARGET_CLIPS = EXPLICIT_N_CLIPS or max(1, int(-(-POPULATION * PERCENT // 100)))
    VERDICT = CW.runtime_verdict(ENVIRONMENT, clips=TARGET_CLIPS)
    print(f"Elegibles {WEAPON}: {POPULATION} | pedido: "
          f"{('%.4g%%' % PERCENT) if PERCENT is not None else str(EXPLICIT_N_CLIPS) + ' clips'} -> {TARGET_CLIPS} clips")
    print(f"Runtime: {VERDICT['device']} | {VERDICT['cores']} vCPU | {VERDICT['ram_free_gb']:.1f} GiB RAM libres")
    print(f"Estimacion: {VERDICT['estimated_seconds_per_clip']} s/clip -> {VERDICT['estimated_hours']} h. {VERDICT['basis']}")
    for message in VERDICT["warnings"]:
        print("AVISO:", message)
    if VERDICT["blocking"]:
        raise RuntimeError("Este runtime no puede completar la seleccion: " + " | ".join(VERDICT["blocking"]))
    PLAN = CW.resource_plan(ENVIRONMENT, pose_profile=POSE_PROFILE, torch_threads=TORCH_THREADS, clips=TARGET_CLIPS)
    SELECTED, SELECTION_INFO = CW.prepare_data_manifest(SOURCE_JSON, WORK, GLOBAL_REGISTRY, weapon=WEAPON,
        max_clips=EXPLICIT_N_CLIPS or TARGET_CLIPS, clip_ids=CLIP_IDS, percent=PERCENT, local_paths=LOCAL_PATHS,
        download_missing=not LOCAL_PATHS, previous_manifest=PREVIOUS_MANIFEST,
        allow_new_registry=CREATE_NEW_GLOBAL_REGISTRY)
    print({key: SELECTION_INFO[key] for key in ("source_records", "weapon_records", "locally_mapped_files", "selected")})
    print("Muestreo:", SELECTION_INFO["sampling"])
    print("Particiones de la muestra:", SELECTED["split"].value_counts().to_dict())
    display(SELECTED[["clip_id", "weapon", "split", "tournament_id", "label_ok"]].head(20))
else:
    print("Modo revision: fija PERCENT_OF_FOIL_CLIPS y ejecuta en Colab, o exporta FENCINGLY_RUN_DATA=1")


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:preflight
# Comprobaciones que tienen que fallar RUIDOSAMENTE, antes de gastar horas de GPU.
# preflight.py viaja junto a este cuaderno; si no esta, subelo o clona el paquete.
import sys, os
from pathlib import Path
for _cand in (Path.cwd(), Path.cwd() / "paquete", Path("/content"), Path("/content/paquete")):
    if (_cand / "preflight.py").is_file() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))
import preflight as PF

PF.protocolo()

# 1. Entorno clavado. VA PRIMERO, ANTES DE LA HUELLA, y el orden no es cosmetico: `_producer()` lee
#    las versiones con `importlib.metadata.version(...)`, o sea los METADATOS EN DISCO, no el modulo
#    cargado. Tras un `pip install numpy==2.5.2` sin reiniciar el runtime, la HUELLA COINCIDE y las
#    medidas se calculan con el numpy equivocado. La comprobacion de huella daria falsa seguridad;
#    esta compara `numpy.__version__` contra los metadatos y levanta.
ENTORNO_PINES = PF.entorno(exigir=True)

# 2. La huella del productor de medidas. producer_sha256 NO es el hash de los 20 ficheros: hashea el
#    diccionario entero, que incluye runtime_versions. Un parche de numpy invalida los 23.091
#    artefactos y el fallo seria un desempate de cache SILENCIOSO.
HUELLA = PF.huella(exigir=True)

# 3. El recorte a la ventana de analisis en el camino POR CLIP. Con la ventana desplegada es inerte
#    (poses.shape[0] == n_frames_window en los 15.452 clips: no hay ni un NPZ de duracion completa).
#    Con almacen ancho pasa de inerte a NECESARIO: sin el, las medidas de guardia se calcularian
#    sobre el clip entero, con desviaciones medidas de hasta 152 grados en fore_ang_guard.
PARCHE = PF.parche_de_recorte(exigir=(VENTANA_ALMACEN != "desplegada"))

# 4. Que la rodaja de entrenamiento CABE en T_max. Si no cabe, el Store se queda los ultimos T_max
#    cuadros de la rodaja y la ventana de guardia [t-4,0 : t-3,5] se pierde EN SILENCIO. El caso
#    6.0/1.0 con T_max=135 da [t-3,47 : t+1,0]: 211 cuadros no caben en 135.
PLAN_RODAJA = PF.plan_rodaja(RODAJA_ENTRENAMIENTO, t_max=T_MAX, exigir=True)

# 5. Y que se sepa lo que cuesta cada cambio, medido y no supuesto.
PF.coste_de_las_listas()


<!-- @@FENCINGLY_DURACION_COMPLETA:porque -->
## Almacenar y entrenar son dos decisiones distintas

**La tabla de abajo NO es una lista de alternativas.** Guardar el clip entero y entrenar sobre una
rodaja son dos decisiones separadas, y tomarlas por separado quita los cuatro precios de la duracion
completa sin renunciar a nada:

| | que cuesta | que compra |
|---|---|---|
| **`VENTANA_ALMACEN = "completa"`** | tiempo de GPU y un runtime High-RAM | la opcionalidad: el clip entero para lo que venga, sin volver a descargar 93 GB |
| **`RODAJA_ENTRENAMIENTO = "recomendada"`** | nada mas | la alineacion, los 488 clips, `T_max` y las etiquetas auxiliares |

Y lo que la duracion completa compra de verdad —re-ventanear las **256 columnas agregadas**, que es
donde esta el techo— se compra igual, porque las agregadas se recalculan sobre la rodaja que se
elija.

**Y la rodaja no cuesta ningun cambio de fuente**, pero NO por envolver `dataset.load_sample`:
eso se probo y ENVENENA EL STORE (`build_store` congela en `store.json` el hash de los bindings de
`dataset.py`, y el Store queda inabrible para siempre). La via correcta es `PF.rodaja_a_disco`:
escribir los NPZ recortados en un directorio nuevo y construir desde ahi, con `load_sample` intacto
de principio a fin. Cuesta ~0,6 GiB de disco para el corpus de florete. Detalle en
la fuente, pero **comprobado que ni el 01 ni el 02 canonicos mencionan `portable_contract`,
`load_bundle` ni `save_bundle` ni una vez**, y el Store se construye en el 01 mientras los bundles se
tocan en el 02, en otra sesion. Sobre un NPZ de ventana desplegada el recorte es un **no-op
verificado** (T = 129 antes y despues).

Lo unico que la rodaja **no** arregla son los 488 clips del QC, porque el QC se congela en la
extraccion: eso queda como via documentada y no tomada, con su cifra, en el documento.

### La aritmetica que decide la rodaja, y hay que verla antes de elegir

Una rodaja `(pre_s, post_s)` a 30 fps ocupa `round(pre_s*30) + round(post_s*30) + 1` cuadros, y el
Store solo guarda `T_max`, **alineados por el final**:

| rodaja | cuadros | `T_max` necesario | cabe en 135 | guardia `[t−4,0:t−3,5]` | 2ª luz |
|---|---:|---:|:---:|:---:|---:|
| `"historica"` 4.0/0.3 | 130 | 135 | **sí** | sí | 99,05 % |
| **`"recomendada"` 4.0/0.6** | **139** | **140** | no | **sí** | **100,00 %** |
| `"ancha"` 6.0/1.0 | 211 | 211 | no | sí | 100,00 % |

**El detalle que hay que ver:** 6,0 + 1,0 = 7,0 s son **211 cuadros**, y `T_MAX = 135` son 4,467 s.
Con `T_max` sin subir, el Store se queda los ultimos 135 de esos 211, o sea **`[t−3,47 : t+1,0]`**, y
la ventana de guardia **se queda fuera igual**: justo lo que la rodaja venia a proteger. **La rodaja
no hace por si sola que `T_MAX` deje de estorbar; hay que subir `T_max` a la vez.**
`PF.plan_rodaja(...)` lo comprueba y levanta.

Subir `T_max` toca `store.T_MAX` y el validador de `TrainingRecipe` (`8 <= T_max <= 135`). Los dos
estan en **ninguna** de las cuatro listas, y `feature_contract.MAX_TIMESTEPS = 256` ya lo permite,
asi que **no se invalida ningun bundle portable**. Lo que si cambia es la entrada de la red, y eso
exige reentrenar y reexportar — que es lo que se va a hacer igual.

**Por que `"recomendada"` y no `"ancha"`:** 4.0/0.6 es la rodaja **minima** que cierra la unica
brecha **medida** —`|dt_ms|` maximo 520 ms, asi que `post_s = 0,6` captura la segunda luz al 100 %—
y conserva la ventana de guardia entera. `"ancha"` tambien funciona, pero **cambia lo que ve la red**
(7 s en vez de 4,3) y **ese cambio no esta medido**: `pre_s = 4,0` se eligio porque 2,5 s recortaba
el inicio de la accion final en el 24 % de los clips, y nadie ha comparado 6,0 contra 4,0 sobre la
red. La atencion es O(T²), asi que 211 cuadros son ~2,5× el coste de atencion de 135. Si se prueba,
que sea como un brazo mas, pareado contra `"recomendada"`, y con el suelo de +0,4 pp del nulo
neuronal delante.

### El reloj es un mando aparte, y es la ganancia mas barata que hay

`USAR_RELOJ = False` lleva la ventana de guardia de **72,78 % a 91,42 %** al **mismo coste exacto**
de decodificacion (129 cuadros, 680 MiB, +0,03 pp de clips bloqueados). No cuesta nada y no lo
compra ninguna otra palanca de la tabla.

**AVISO:** hay trabajo en curso para **conectar** la tabla del reloj. Las dos decisiones interactuan,
asi que hay que medirlas **juntas, en un factorial 2×2**, no una encima de la otra. Si el otro cambio
ya esta dentro cuando se ejecute esto, hay que declarar las dos configuraciones junto al resultado.

---

## Por que duracion completa, y cual es su precio medido

Las seis razones de la decision, y al lado lo que cada una cuesta o gana **medido**, no supuesto.
Todas las cifras salen de reevaluar los 15.451 artefactos de QC ya escritos del corpus, sin
re-extraer nada.

| razon | lo que se gana | medido |
|---|---|---|
| 1. `POST_S = 0,3` deja la segunda luz fuera en el 3,12 % de los clips | la captura entera | `\|dt_ms\|` maximo **520 ms** sobre los 3.331 clips de dos luces (p50 80, p95 267, p99 360; 100 % < 1.000 ms). **`post_s = 0,6` ya la captura entera.** La duracion completa no anade nada aqui |
| 2. el bloque de 256 columnas agregadas esta congelado y lleva el atajo casi entero | poder re-ventanearlo | **esto si lo exige la duracion completa.** Re-ventanear solo el bloque temporal no movio nada (media +0,30 real contra +0,27 de su nulo, exceso +0,03) |
| 3. la ventana de guardia `[t-4,0 : t-3,5]` no existe en el 22,88 % de los clips | tenerla siempre | pasa de **72,78 % a 91,42 % solo apagando el reloj**, al mismo coste exacto. Y su techo es **97,04 %, no 100 %**: en ~3 % de los clips el tocado ocurre a menos de 4,0 s del comienzo. La duracion completa **no puede inventar cuadros que nadie filmo** |
| 4. el transformador de video fallo por submuestreo (16 cuadros / 1,2 s = 75 ms de paso, y el suceso decisivo es un desfase de 10-30 ms) | muestreo denso cerca del tocado | lo exige la duracion completa, pero es una via **cerrada** por medida (VideoMAE no llego al suelo de 0,8 pp y predice la sede al 97 %) |
| 5. la tabla del reloj y la deteccion de repeticiones | cuadros anteriores a la frase | `pre_s = 6,0` los da en el 97,04 % |
| 6. la estatura gana cuadros de guardia | su escala | igual que 3 y 5 |

### El precio, en la misma unidad

| candidato | cuadros mediana | MiB p95 | rechazados T4 | rechazados High-RAM | QC bloq | 2a luz | guardia |
|---|---:|---:|---:|---:|---:|---:|---:|
| 4.0/0.3 + reloj (**hoy**) | 129 | 1.554 | 0,00 % | 0,00 % | 5,53 % | 99,05 % | 72,78 % |
| 4.0/0.3 sin reloj | 129 | 1.554 | 0,00 % | 0,00 % | 5,56 % | 99,05 % | **91,42 %** |
| **6.0/1.0 sin reloj** | **210** | 2.515 | 0,00 % | 0,00 % | 6,52 % | **99,99 %** | **97,04 %** |
| 8.0/3.0 sin reloj | 294 | 3.939 | 4,36 % | 0,49 % | 8,30 % | 99,99 % | 97,04 % |
| **DURACION COMPLETA** | **318** | 5.233 | **11,31 %** | **0,50 %** | **8,69 %** | 99,99 % | 97,04 % |

**Los cuatro precios de la duracion completa. Los tres ultimos los quita la rodaja; el primero no.**

0. **ROMPE LA ALINEACION, y es el peor.** `build_store` alinea por el FINAL del array y
   `feature_contract` lo declara (`alignment: right`, `truncation: tail`). Con un NPZ de duracion
   completa el array termina al final del **CLIP**, y el clip sigue despues del tocado: la cola tras
   el tocado tiene **mediana 2,33 s** (p05 0,93, media 2,62, p95 5,23, max 14,57) sobre 15.453 clips
   [medido], y supera 1,0 s en el **93,70 %**. Asi que el Store guardaria `[t−2,17 : t+2,33]` en vez
   de `[t−4,2 : t+0,3]`: **el ultimo cuadro deja de ser el tocado** y la **ventana de guardia
   desaparece del Store en ~94 % de los clips** — justo lo que la duracion completa venia a
   garantizar. No salta ningun error. **Esto es una regresion, no un empate**, y es la razon de que
   la rodaja sea obligatoria y no una optimizacion. `PF.alineacion_del_store` lo mide sobre el Store
   construido y **levanta**.

1. **La duracion completa exige un runtime High-RAM.** Con el tope de `max_decoded_bytes` que
   `resource_plan` calcula (`min(6,0 GiB, RAM_libre x 0,35)`), un T4 estandar rechaza el **11,31 %**
   de los clips y el rechazo esta **correlacionado con la resolucion**, o sea que sesga la seleccion:
   es exactamente el fallo que el runbook del 2026-09-06 elimino y que la duracion completa reabre.
   En High-RAM el rechazo baja al **0,50 %**. La frase del runbook *"You no longer need High-RAM to
   avoid resolution bias"* **deja de ser cierta con duracion completa**.
2. **La duracion completa cuesta 488 clips de entrenamiento (3,16 %).** `clip_quality` marca
   `cut_in_window` y `replay_in_window` como **bloqueantes** (peso 0,0) y `build_current_store`
   descarta toda fila con `qc_blocking` no vacio. Al ensanchar la ventana entran cortes de camara que
   hoy quedan fuera: 828 clips (5,36 %) tienen al menos un corte. Bloqueados 853 (5,52 %) -> **1.341
   (8,68 %)**; entran al Store 14.598 -> **14.110**. Las repeticiones **no** son el problema (3 clips).
   **Y la rodaja NO lo arregla sola**, porque el QC se calcula en la EXTRACCION sobre la ventana
   GUARDADA y queda congelado en `analysis/qc/<clip>.json`; elegir una rodaja despues no lo cambia.
   Se recupera acotando esas dos banderas a la ventana de analisis en `clip_quality._flatten_vm`
   (via documentada y no tomada, con su cifra, en el documento). `clip_quality.py` solo esta en
   `FINGERPRINT_SOURCES`, asi que el coste es re-cachear el panel de coach, no re-materializar los
   23.091 artefactos.
3. **Las etiquetas auxiliares se apagan en silencio.** `store_truncated = prepared_n_frames > T_max`
   pasa a True en casi todos los clips, y `attach_aux` (`store.py:244`) y `Store` (`store.py:417`)
   exigen `not store_truncated` para admitir una etiqueta: quedan a codigo -1, mask 0, conf 0 **sin
   aviso**. Hoy es 0 de 14.437 muestras [medido]. **La rodaja lo arregla**, porque una rodaja que
   cabe en `T_max` no trunca. `PF.alineacion_del_store` lo cuenta.
4. **La red no ve mas contexto por la duracion completa.** El Store guarda `T_max` cuadros alineados
   por el final, asi que lo que ve la red lo decide la **rodaja**, no el almacen. Con
   `"recomendada"` son 139 cuadros (4,63 s) contra los 130 (4,33 s) de hoy: casi lo mismo, y a
   proposito. Subir `feature_contract.MAX_TIMESTEPS` (256) **no hace falta** para ninguna de las tres
   rodajas, y conviene no tocarlo: esta en `portable_contract.SEMANTIC_SCOPE` con alcance de modulo
   completo, asi que cambiarlo mueve `configuration_sha256` e **invalida todos los bundles portables**
   (sintoma: `ContractError: Portable: semantica AST/configuracion no conforme`, la app corre sin
   modelo y `judge` responde `model_unavailable`).

   Traducido: **la duracion completa es una inversion en el camino de MEDIDAS (bloque agregado,
   guardia, estatura, reloj) y en no tener que volver a descargar. No es una mejora del modelo
   secuencial, y no hay que esperar que el numero suba por ella.**


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:plan_ventana
# Que va a pasar con la ventana de ALMACEN en ESTE runtime concreto: cuadros y bytes por clip, y que
# fraccion del corpus rechazaria el planificador de decodificacion. Un rechazo correlacionado con
# la resolucion SESGA la seleccion, asi que se mira antes, no clip por clip despues de horas.
if RUN_DATA:
    print(PF.BARRIDO_MEDIDO)
    PLAN_VENTANA = PF.plan_ventana(ENVIRONMENT, VENTANA_ALMACEN)
    if PLAN_VENTANA["fraccion_rechazada_estimada"] > 0.005 and VENTANA_ALMACEN == "completa":
        print()
        print("DECISION: este runtime no es High-RAM suficiente para duracion completa sin sesgo.")
        print("Opciones, por orden de preferencia:")
        print("  a) Entorno de ejecucion -> Cambiar tipo de entorno -> GPU + High-RAM, y reejecutar")
        print("  b) VENTANA_ALMACEN = 'amplia' (6.0/1.0): no rechaza nada en ningun runtime, y la")
        print("     rodaja de entrenamiento no cambia, asi que el numero sigue siendo comparable")
        print("     entre las dos: es la MISMA cadena de medida con distinto almacen")
        print("  c) subir a mano el tope de 6,0 GiB de resource_plan, sabiendo que el pico REAL")
        print("     medido es 1,08x una copia BGR (1,66 GiB para 1,54 GiB), no 2x")
    print()
    print(PF.COLA_TRAS_TOCADO)


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:respaldo
# RESPALDO INCREMENTAL. Una corrida de horas en Colab se muere sola, asi que esto no es un adorno.
#
# Destino: repositorio PRIVADO de Hugging Face. La excepcion esta escrita en AGENTS.md linea 4,
# decidida por el usuario el 2026-09-09 y acotada: privado SOLO, artefactos de extraccion SOLO.
# Un repo PUBLICO sigue prohibido, porque los recortes son fotogramas reales de retransmision y el
# manifiesto lleva nombres de tiradores, torneos y resultados.
print(PF.RESPALDO)

# El token es una CREDENCIAL: de un secreto de Colab o de HF_TOKEN, nunca en el cuaderno. Sin el,
# esto AVISA A GRITOS y sigue: la corrida no sera reanudable entre sesiones.
RESPALDO = PF.aviso_respaldo(exigir=False)
REPO_PRIVADO = os.environ.get("FENCINGLY_HF_REPO", "")     # p.ej. "usuario/fencingly-extraccion"
RESPALDO_DIR = str(Path(WORK).parent / "_respaldo")

# El tamano de lote se DERIVA del ritmo medido, no se fija a ojo. Medido en la RTX 3060 con
# process_workers=1 y ventana completa: 17,2 s/clip en frio. Con una instantanea cada 10 min salen
# 35 clips por lote. Y se agrupa en `tar` porque la plataforma admite < 100.000 ficheros por repo y
# < 10.000 por carpeta: con 34.374 clips y varios ficheros cada uno, sin agrupar se roza el limite.
SEGUNDOS_POR_CLIP = float(os.environ.get("FENCINGLY_S_POR_CLIP", "17.2"))
CLIPS_POR_LOTE = PF.clips_por_lote(SEGUNDOS_POR_CLIP)
print(f"lote de respaldo: {CLIPS_POR_LOTE} clips ({SEGUNDOS_POR_CLIP:g} s/clip medidos)")

# Y lo que hay hecho DE VERDAD, contado por el indice y sus NPZ presentes, no por un contador:
# si el contador dice 1.200 y el indice tiene 1.180, la reanudacion miente.
ESTADO = PF.estado_reanudacion(WORK, drive_dir=RESPALDO_DIR)
PENDIENTES_RESPALDO = PF.respaldo_pendientes(WORK, RESPALDO_DIR)
print(f"pendientes de empaquetar: {len(PENDIENTES_RESPALDO)}")


## FIE: precargar sólo lo que vas a consultar, con procedencia

La extracción y el entrenamiento no dependen de una conexión FIE. Si tienes la caché entregada, fija `FIE_CACHE_SOURCE`
y se importan sus bytes con SHA-256 y fecha de consulta original a `WORK/fie_tournaments_cache/`, sin modificar el origen.
La caché local auditada contenía 45 respuestas; ese número describe esa instantánea, no una cuota universal ni una descarga
que se vaya a repetir. El release puede incluir otra selección explícita de respuestas.

Para ampliar, activa **ambos** `RUN_FIE_PREFETCH=True` y `FIE_ONLINE=True`, y rellena eventos como
`{"season": ..., "tournament_id": ..., "competition_id": ...}` con IDs verificados. Un nombre local de torneo, un año o un
marcador parecido no identifican por sí solos un evento FIE. `VERIFIED_FIE_IDS` sólo admite mappings revisados: por ejemplo,
los IDs auditados Volpi 19878 y Kiefer 21717 no autorizan a resolver automáticamente otros nombres ambiguos. Los perfiles
se priorizan por frecuencia entre los clips seleccionados, hasta ocho IDs, sin scrapear todos los atletas.

Se reutiliza `FIETournamentClient`: un consumidor, intervalo global de proceso mínimo **0,65 s**, hasta 40 peticiones y
paginación limitada a cuatro páginas por endpoint en esta receta. No se descargan PDFs ni vídeos desde FIE. Un cache miss
o un límite de páginas queda en `errors`, nunca como un cero deportivo. Las respuestas conservan URL, parámetros,
`fetched_at`, estado HTTP y hash. No se unen sénior/júnior, individual/equipos ni temporadas como si fuesen el mismo universo.
Un ranking actual o mejor ranking de carrera no es un ranking preasalto; la fecha del evento tampoco da el timestamp de
cada tocado. La caché sirve a la web como evidencia, **no como nuevas etiquetas de entrenamiento**.


In [ ]:
if RUN_DATA and FIE_CACHE_SOURCE:
    print(CW.seed_fie_cache(FIE_CACHE_SOURCE, WORK / "fie_tournaments_cache"))
if RUN_FIE_PREFETCH:
    fencer_ids = CW.top_fencer_ids(SELECTED if SELECTED is not None else [], VERIFIED_FIE_IDS, limit=8)
    fie_report = CW.prefetch_fie(WORK, events=FIE_EVENTS, fencer_ids=fencer_ids, online=FIE_ONLINE,
                               max_requests=40, max_pages=4)
    print({"requests": fie_report["requests"], "items": len(fie_report["items"]), "errors": fie_report["errors"]})
else:
    print("FIE sin nuevas consultas. Las cachés ya presentes viajarán en el workspace")


## Extracción acotada y checkpoints: qué se ejecuta de verdad

El plan inspecciona GPU, RAM, CPU, disco y ffmpeg; **no** supone que «GPU disponible» signifique «VRAM suficiente».
La cola paralela descarga como máximo los paths que caben en `queue_size` (1–8), con 1–4 workers. Sólo hay **una consumidora
GPU** y una ventana decodificada a la vez: no se apilan todos los vídeos ni se envían seis decodificaciones simultáneas.
Antes de leer los frames se comprueban duración, resolución y un presupuesto conservador de bytes de ventana con margen.
Un presupuesto insuficiente produce un error explícito; no se reduce la calidad ni se inventan píxeles para ocultarlo.

YOLO usa `half=True` únicamente en CUDA; en CPU usa FP32. Necesita un fichero de pesos que ya hayas obtenido legítimamente:
no se descarga un modelo al importarlo. Un OOM CUDA reduce el lote por mitades hasta uno y reintenta esa etapa. Bajar el
lote no reduce la RAM de vídeo: si el fallo es RAM, baja la selección/presupuesto de vídeo y usa un nuevo plan apropiado.
Las estimaciones son márgenes de planificación, no mediciones de velocidad ni una garantía contra el cierre de Colab.

**Perfil explícito de pose:** `balanced960` pide imgsz=960 y `fast640` pide imgsz=640. Elige según presupuesto y revisión,
no según qué altura/resultado te guste más. El seguimiento metrológico halló dos casos con menores errores anatómicos a
960, pero no una ganancia universal ni una validación del detector. El wrapper no cambia el perfil automáticamente ni
retoca umbrales de pose. El presupuesto de decodificación sigue correspondiendo al vídeo original; cambiar imgsz no crea
nuevos frames ni cambia sus tiempos.

**Hilos efectivos, no sólo una variable de entorno:** Ultralytics puede fijar Torch a tres hilos durante inicialización o
primer predict aunque hayas pedido uno antes. El wrapper mide y restaura el presupuesto después de inicializar y en cada
frontera de `predict`, sin modificar `pose.py` ni funciones globales del detector. Guarda `observed`, `effective_after`,
`drift_events` y `max_observed`; si observó tres, NO dice que toda esa inferencia fuese CPU1. Las mediciones de frontera
no demuestran que internamente jamás cambiase el número de hilos. No se promete un pico continuo que no se ha medido.

Las etapas nativas calculan overlay → ventana `[max(allez, luz−4 s), luz+0,3 s]` → pose/tracking → paneo/duplicados → NPZ.
El QC usa la ventana **efectivamente extraída**, no una ventana antigua de 2,5 s. También guarda medidas de vídeo
(cortes/replays/movimiento) y payloads de movimiento para React. Si falta audio, se declara ausente; silencio/missing no
se transforma en un contacto conocido. Después de la GPU, la CPU ejecuta los cinco módulos nativos de coaching mediante
`CP._one_clip`: clip_stats, pies, brazo, distancia y continuidad. Sus filas L/R se materializan para no repetir todo el
cálculo al abrir cada perfil en la web. YOLO, QC, audio y anclas siguen siendo detectores no validados independientemente.

Cada clip sólo queda `validated` cuando el NPZ completo, tiempo, formas, versiones, sidecars y sus hashes pasan y se ha
persistido el checkpoint. Una existencia de archivo mayor que cero no basta. Las filas fallidas quedan trazables.
`CHECKPOINT_EVERY` copia snapshots completos a Drive sólo si lo configuraste; el journal local es por clip. Aumenta la
frecuencia de intervalo (por ejemplo 100) al ampliar el corpus para no acumular demasiados ZIP completos. El disco libre
se comprueba también al exportar. No se borran snapshots históricos automáticamente.

### Borrado opt-in, estrecho y demostrable

`DELETE_OWN_DOWNLOADED_MP4=False` conserva todo. Si lo activas, sólo se puede eliminar el **MP4 temporal creado por una
descarga de esta llamada** en su subcarpeta única. Se verifica que no cambió, que el NPZ/checkpoint/sidecars siguen íntegros
y, si hay callback de snapshot, que éste terminó bien. Nunca se elimina un MP4 preexistente, una importación local, un
fichero de Drive, pesos, datos históricos o una descarga fallida. Una respuesta `cached` tampoco autoriza borrado.
La eliminación exige ahora los tres archivos canónicos de QC, moments y measurements en la evidencia del ledger,
además del NPZ y checkpoint íntegros. Un fallo de materialización de coaching, un sidecar ausente o desactivar alguno de
esos pasos impide borrar el MP4. No se usa la opción legacy de borrado de `pipeline.process_many`.


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:extraccion
EXTRACTION_RESULT = None
if RUN_DATA:
    from dataclasses import replace
    BUDGET = replace(CW.RunBudget(**PLAN["budget"]), checkpoint_every=CW.checkpoint_interval(len(SELECTED)))
    CW.write_json(WORK / "run_config.json", {"environment": ENVIRONMENT, "plan": PLAN, "budget": BUDGET.__dict__,
                 "weapon": WEAPON, "post_touch_only": True,
                 "ventana_almacen": VENTANA_ALMACEN, "pre_s": PRE_S, "post_s": POST_S,
                 "use_clock": USE_CLOCK, "usar_reloj": USAR_RELOJ,
                 "rodaja_entrenamiento": RODAJA_ENTRENAMIENTO,
                 "rodaja_pre_s": RODAJA_PRE_S, "rodaja_post_s": RODAJA_POST_S,
                 "t_max": T_MAX, "plan_rodaja": PLAN_RODAJA,
                 "protocolo": "linea_base_nueva_no_comparable_con_73_88",
                 "pines": ENTORNO_PINES["observado"], "huella": HUELLA["medidas"], "release": RELEASE_PROVENANCE,
                 "requested_percent": PERCENT, "runtime_verdict": VERDICT,
                 "sampling": SELECTION_INFO["sampling"]}, overwrite=True)
    if not POSE_WEIGHTS and IN_COLAB:
        POSE_WEIGHTS = str(RUN_PARENT / "yolo11m-pose.pt")
        if not Path(POSE_WEIGHTS).is_file():
            print("Descargando los pesos YOLO-pose desde", POSE_WEIGHTS_URL, "(Ultralytics, AGPL-3.0)", flush=True)
            CW.download_url(POSE_WEIGHTS_URL, POSE_WEIGHTS)
        print("Pesos de pose:", POSE_WEIGHTS)
    extractor = CW.create_extractor(POSE_WEIGHTS, PLAN)
    processor = CW.NativeProcessor(extractor, pre_s=PRE_S, post_s=POST_S, use_clock=USE_CLOCK,
                               audio=True, qc=True, web_moments=True, measurements=True)
    def snapshot_data(work, run_id, completed):
        return CW.publish_workspace(work, EXPORT_DIR, split_registry=GLOBAL_REGISTRY, drive_dir=DRIVE_DIR,
                    metadata={"phase": "data_checkpoint", "run_id": run_id, "completed": completed})
    EXTRACTION_RESULT = CW.run_data(SELECTED, WORK, processor, local_paths=LOCAL_PATHS,
        download_missing=not LOCAL_PATHS, delete_downloads=not LOCAL_PATHS, budget=BUDGET,
        on_checkpoint=snapshot_data if DRIVE_DIR is not None else None)
    print({key: EXTRACTION_RESULT[key] for key in ("processed", "resumed", "n_ok", "n_failed", "one_gpu_consumer")})
    if EXTRACTION_RESULT["n_failed"]:
        print("Motivos de fallo, contados y visibles:", EXTRACTION_RESULT["failure_reasons"])
    THREAD_DIAGNOSTICS = getattr(extractor, "thread_budget_diagnostics", None)
    if THREAD_DIAGNOSTICS is not None:
        CW.write_json(WORK / "checkpoints" / ("pose_threads_" + CW.stamp() + ".json"), THREAD_DIAGNOSTICS)
        print("Hilos observados/corregidos, no un pico continuo certificado:", THREAD_DIAGNOSTICS)
    del extractor, processor
    from fencingly_v2 import resources as RS
    RS.free_memory()
else:
    print("No se cargo YOLO, no se decodificaron videos y no se ejecutaron descargas")


## Qué guardamos y qué significan tiempo, máscaras y unidades

| Artefacto | Contenido | Lectura correcta |
|---|---|---|
| `npz/<clip_id>.npz` | Poses T×2×17×3, cajas, source, luces, audio opcional, pan, dup, time_s, light_observed y metadata | Raw derivado del detector; no son coordenadas 3D ni verdad humana |
| `analysis/qc/` | QC y medidas del vídeo, con flags/bloqueos y ventana efectiva | Cobertura/calidad heurística no equivale a accuracy |
| `analysis/moments/` | Payloads del mismo builder que consume la web, con binding de caché | Tiempos/series con gaps, máscaras por articulación y semántica del ancla; sin interpolación nueva |
| `analysis/measurements/` | Dos filas escalares L/R calculadas por los cinco módulos de coaching | Materializadas por el productor; no llamarlas recalculadas en el runtime importador |
| `store/` | Arrays memmap preparados, metadata, tiempos y masks | Vista de entrenamiento derivada; NPZ original se conserva |
| `checkpoints/clips/` | Identidad, hashes de NPZ/evidencia y hash del MP4 de entrada | Evidencia de finalización/reanudación, no permiso de borrado importado |
| `provenance/` | Snapshot explícito del registro GLOBAL | Trazabilidad de particiones; no autoridad instalada automáticamente |
| `fie_tournaments_cache/` | Respuestas inmutables y metadata de consulta | Fuentes publicadas, fecha y alcance; no etiquetas ocultas en X |

### Importar medidas ya calculadas, sin repetir todos los NPZ del perfil

El JSON de `measurements` liga las filas al SHA-256 del NPZ y a metadata semántica: nombres, actores, sexo/género declarado,
arma, tocado, tanteos, luces/cobertura, formato/alcance y calibración explícita. Normaliza None/nan/unknown y las variantes
numéricas de CSV; no usa rutas absolutas, mtime ni el directorio de instalación para decidir identidad. Mover el workspace
no cambia sus medidas. Añadir después FIE/formato/altura/escala no autoriza reatribuir silenciosamente scorers o calibración:
si cambia un campo ligado, la lectura devuelve un miss explícito y debe generarse otra versión con la metadata correcta.

El productor conserva método/esquema, hashes AST de fuentes seleccionadas por nombre relativo, parámetros, versiones y
unidades. `reviewed=False` es un booleano real; una revisión declarada por el usuario tampoco convierte estas estimaciones
en gold humano independiente. Un desconocido se guarda como null, no como cero. No se completan huecos entre predicciones.

`measurement_artifacts.load_measurements(WORK, clip_id, npz_path, row)` sólo lee JSON acotado y verifica bytes del NPZ;
no llama a `CP._one_clip`, `CP.build_video_stats`, extractores ni vídeo. Devuelve status hit/miss, reason, rows y provenance.
Un hit significa **producer_materialized**, siempre `recomputed_current=False`. Para exigir el método de una versión,
calcula `measurement_producer()` una vez y pasa `expected_producer`; si difiere, se rechaza salvo una lectura histórica
explícita `allow_historical=True`, que conserva el método original y no se anuncia como recálculo actual.

El lector `load_cached_motion` aplica los mismos controles y mantiene `anchor_semantics`, las máscaras por articulación,
los segmentos y los tiempos de `fencingly.motion.v1`. No sustituye `video_url`: la aplicación asocia su recurso actual después
de validar. Un motion antiguo que sólo guardase el hash del NPZ y no ligase la metadata no se reutiliza como equivalente.
Las corrupciones generan un error visible; no se disfrazan de medidas recalculadas ni de cobertura cero.

Si ya tienes NPZ actuales válidos y sólo falta materializar coaching, no hace falta volver a extraer vídeo: usa
`export_measurements(D.load_sample(npz_path), row, WORK, npz_path=npz_path)` por clip, con el row de índice correspondiente.
Se valida que la muestra realmente procede de ese NPZ. Un archivo igual se reutiliza; uno distinto NO se pisa: conserva la
versión anterior y elige una nueva carpeta de resultados. Los NPZ legacy sin linaje temporal actual siguen bloqueados.
La caché FIE continúa viajando aparte como respuestas públicas en bruto, sin red al importarla y sin convertirla en targets.

Las coordenadas de pose raw son `x/W, y/H` con tamaño de imagen explícito. El paneo tiene convención declarada. Los rasgos
anatómicos usan escalas de torso/pierna; los píxeles o fracciones de anchura no se renombran como metros. Una estimación
métrica por cuerpo/cámara sigue sin ser calibración independiente.

`time_s` actual procede de índices/fps del contenedor; el pipeline declara `container_fps_assumed`, no PTS reales del
sensor ni precisión subframe. Preparar a 30 fps no crea observaciones físicas nuevas. Duplicados, cortes, freezes y huecos
no se comprimen temporalmente. `observed` identifica soporte realmente observado; `supported` admite sólo interpolación
corta respaldada por endpoints del mismo segmento (hasta 0,2 s según contrato); los frames/articulaciones sin soporte
siguen inválidos. Las máscaras de luces son independientes de los duplicados de pose. No se añade interpolación al vídeo
ni al replay para fabricar tiempo, píxeles o contacto de hoja.

### Store elegible, no mezcla de históricos

La celda siguiente aplica el merge global nativo, filtra etiqueta/coherencia/cobertura/QC y requiere train/val/test con
cero cruces **conocidos**. Con cuatro clips puede no existir un split utilizable: eso es un resultado honesto, no una razón
para pasar validación como test. Amplía la selección por grupos antes de entrenar. El Store se crea en destino nuevo.
Los NPZ legacy con tiempos insuficientes se rechazan; no se concatenan los 962 NPZ v2 y los seis actuales para inflar N.
Si falta un split, aún puedes exportar la evidencia para inspección, pero no anunciar una evaluación completa.


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:store
STORE_RESULT = None
STORE_ERROR = None
if RUN_DATA:
    INDEX = CW.finalize_data_index(WORK, GLOBAL_REGISTRY, weapon=WEAPON)
    print("Filas de proceso:", len(INDEX))
    if BUILD_STORE:
        try:
            # El Store se construye desde los NPZ YA RECORTADOS a la rodaja.
            #
            # NO se envuelve `dataset.load_sample` en caliente: se probo y ENVENENA EL
            # STORE. `build_store` graba en `store.json` el `runtime_compatibility()` del
            # momento, que hashea los bindings de `dataset.py`; con el envoltorio puesto
            # queda congelado un hash que ya no coincide y el Store NO SE PUEDE ABRIR
            # NUNCA MAS (ContractError: Bindings runtime incompatibles: dataset.py).
            # Medido: el Store salio con la alineacion perfecta y aun asi inabrible.
            # Ver PF.ENVOLTORIO_PROHIBIDO.
            #
            # `construir_store_con_rodaja` replica el embudo de `build_current_store`
            # (que fija npz_dir a work/npz y por eso no sirve), conserva sus dos guardas
            # -- validate_splits y el eligible_index que no se reescribe -- y anade
            # `embudo_elegibles`, que nombra el paso que vacia la seleccion en vez de
            # levantar KeyError('coverage_L') despues de horas de GPU.
            STORE_RESULT = PF.construir_store_con_rodaja(
                WORK, GLOBAL_REGISTRY, rodaja_nombre=RODAJA_ENTRENAMIENTO,
                t_max=T_MAX, weapon=WEAPON)
            print(STORE_RESULT)
        except ValueError as error:
            STORE_ERROR = str(error)
            print("Store NO certificado para entrenamiento:", STORE_ERROR)
            print("Se conserva la evidencia; no se sustituye test ni se cambian los splits")


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:verificacion
# Verificacion de la corrida. Nada de esto se supone: se cuenta.
if RUN_DATA:
    # 1. El indice, contado con el LECTOR REAL. AnalysisRepository.index() se salta EN SILENCIO un
    #    directorio de NPZ sin npz/index.csv (costo 689 clips invisibles), y valid_id exige
    #    [A-Za-z0-9_-], asi que los 13 clips con id no ASCII (Vicuna, Kauai'O'o, Vendeen) se caen
    #    aunque su fila este escrita. No relajes el validador: es una brecha conocida y aceptada.
    INDICE_REAL = PF.indice_real(WORK)

    # 2. El coste de QC de esta ventana, medido en ESTA corrida y no heredado del corpus.
    import json as _json
    from pathlib import Path as _Path
    _qc = sorted((_Path(WORK) / "analysis" / "qc").glob("*.json"))
    _n = _bloq = _corte = _rep = 0
    _motivos = {}
    for _p in _qc:
        try:
            _q = _json.loads(_p.read_text(encoding="utf-8"))
        except Exception:
            continue
        _n += 1
        _b = list(_q.get("blocking") or [])
        _bloq += int(bool(_b))
        _corte += int("cut_in_window" in _b)
        _rep += int("replay_in_window" in _b)
        for _f in _b:
            _motivos[_f] = _motivos.get(_f, 0) + 1
    print(f"\nQC de esta corrida: {_n} clips evaluados")
    if _n:
        print(f"  bloqueados (los que build_current_store descarta): {_bloq} ({100*_bloq/_n:.2f} %)")
        print(f"    de ellos por cut_in_window   : {_corte} ({100*_corte/_n:.2f} %)")
        print(f"    de ellos por replay_in_window: {_rep} ({100*_rep/_n:.2f} %)")
        print("  desglose:", dict(sorted(_motivos.items(), key=lambda kv: -kv[1])))
        print("  Referencia con la ventana desplegada sobre los 15.451 del corpus: 5,52 % bloqueados,")
        print("  cut_in_window 2,05 %. Con duracion completa: 8,68 % y 5,36 %. Si tu corrida se")
        print("  parece a la segunda columna, es la ventana, no un fallo de extraccion.")

    # 2.bis LA TABLA DEL RELOJ. Coste cero y es la ganancia mas grande de la noche:
    #       `piste_abs.allez_anchor` (piste_abs.py:216) hace
    #           clk = parse_list(_get(row, "clock_running_s")) or meta.get("clock_running_s")
    #       o sea que lee LA FILA DEL INDICE ANTES QUE EL META. Asi que el reloj le llega por
    #       npz/index.csv sin reextraer, sin NPZ nuevos y SIN CAMBIAR LA HUELLA. Y la columna ya la
    #       emite el conducto: es la 45 de las 119. Esto no es un cambio que hacer, es un hecho que
    #       COMPROBAR, y de paso cuenta a cuantos clips desbloquea x_abs / dist_m / dist_allez_*_m,
    #       que hoy son NaN en el 100 % de las extracciones.
    RELOJ_INDICE = PF.columna_reloj(WORK)

    # 3. LA PUERTA DE ALINEACION. Es la comprobacion mas importante del cuaderno.
    #
    #    `build_store` alinea por el FINAL del array y `feature_contract` lo declara
    #    (`alignment: right`, `truncation: tail`). Con un NPZ de duracion completa el array termina
    #    al final del CLIP, no del tocado, y la cola tras el tocado tiene mediana 2,33 s (p95 5,23,
    #    max 14,57) sobre 15.453 clips medidos. Sin la rodaja aplicada, el Store guardaria
    #    [t-2,17 : t+2,33] en vez de [t-4,2 : t+0,3]: el ultimo cuadro deja de ser el tocado y la
    #    ventana de guardia [t-4,0 : t-3,5] desaparece en ~94 % de los clips. SIN NINGUN ERROR.
    #
    #    Esto lo mide sobre el Store ya construido y LEVANTA si esta desalineado. Funciona sin
    #    aplicar ningun parche: usa solo campos que `sample_meta.jsonl` ya escribe. Verificado
    #    sobre un Store real de ventana desplegada: mediana 0,267 s contra post_s 0,3, y
    #    store_truncated=True en 0 de 14.437 muestras.
    if STORE_RESULT is not None:
        ALINEACION = PF.alineacion_del_store(_Path(WORK) / "store",
                                             post_s_esperado=RODAJA_POST_S, exigir=True)
        from fencingly_v2 import store as _ST, feature_contract as _FC
        from fencingly_v2 import training_recipe as _TR
        print(f"\n  topes: store.T_MAX={_ST.T_MAX} | TrainingRecipe valida 8<=T_max<=135"
              f" | feature_contract.MAX_TIMESTEPS={_FC.MAX_TIMESTEPS}")
        print(f"  T_max pedido {T_MAX} para una rodaja de {PLAN_RODAJA['cuadros']} cuadros")
        if T_MAX > 135:
            print("  !! T_max > 135 exige subir store.T_MAX y el validador de TrainingRecipe.")
            print("     Los dos ficheros estan en NINGUNA de las cuatro listas protegidas y")
            print("     MAX_TIMESTEPS=256 ya lo permite, asi que NO se invalida ningun bundle.")
            print("     Se sube en store.T_MAX y en el validador de TrainingRecipe. Si no se")
            print("     construyo con T_max=135 y la puerta de arriba lo habria dicho.")


## Exportar y pasar al notebook 02 — éste es el archivo fundamental

Se genera `fencingly_workspace_<fecha>.zip` con un manifiesto `fencingly.workspace.v1`, inventario de SHA-256 y datos
allowlisted. Incluye manifiestos/NPZ/Store/análisis/caché FIE/checkpoints; **no incluye vídeos originales, código de usuario,
entornos, dependencias ni ZIPs arbitrarios**. El ZIP de datos no ejecuta código al restaurarse. Se valida completamente
antes de extraer a una carpeta nueva, con límites de tamaños/expansión, sin traversal, drives/ADS, symlinks, colisiones ni
sobrescrituras. Los modelos se cargan sólo después por el loader de confianza; extraer no certifica compatibilidad.

Guarda la ruta y el SHA. Abre `02_Training.ipynb` del mismo release y dale este workspace. Si sólo querías inspeccionar
clips/QC/movimiento/FIE sin modelo, puedes importar el workspace de datos en la web local; seguirá siendo evidencia
retrospectiva con los límites visibles. No hagas pasar un Store ausente o fallido por un dataset listo.


In [ ]:
DATA_EXPORT = None
if RUN_DATA:
    DATA_EXPORT = CW.publish_workspace(WORK, EXPORT_DIR, split_registry=GLOBAL_REGISTRY, drive_dir=DRIVE_DIR,
        browser_download=DOWNLOAD_FINAL_IN_BROWSER, metadata={"phase": "data", "weapon": WEAPON,
        "store_ready": STORE_RESULT is not None, "store_error": STORE_ERROR, "independence": "not_established"})
    print("ZIP para 02/web:", DATA_EXPORT["path"])
    print("SHA-256:", DATA_EXPORT["sha256"])
    print("Entrada a 02: FENCINGLY_WORKSPACE_ZIP =", DATA_EXPORT.get("drive_path", DATA_EXPORT["path"]))
else:
    print("No se exportó un dataset inexistente. Activa RUN_DATA tras revisar la configuración")


## Reanudar / resolver un error sin destruir los originales

- Si se desconecta Colab, abre el mismo release, restaura el último ZIP en **otro WORK nuevo**, apunta al registro GLOBAL
persistente de Drive y repite la selección. `run_data` verifica hashes/versiones y omite exclusivamente clips validados.
No descarga de nuevo un clip reanudable sólo porque ya no exista el MP4 temporal.
- Un NPZ sin checkpoint puede ser una extracción interrumpida: no se adopta por existir ni se sobrescribe. Conserva esa
carpeta para revisión o elige otra versión; no cambies su metadata para fingir una extracción actual.
- Si cambió un MP4, un sidecar, el JSON fuente o el contrato, usa una versión nueva y documenta la procedencia. No borres
los resultados que explican por qué falló la primera ejecución.
- HTTP 403/429 no autoriza eludir permisos. Usa clips locales autorizados, espera según el proveedor o solicita acceso.
- Si el registro denuncia un cruce de particiones, arregla las identidades/historia con evidencia, fuera del entrenamiento.
- Los tests de esta entrega usan CPU y fixtures para errores/OOM/Colab. Las ramas CUDA/T4 y Drive real deben probarse en
el runtime que vayas a usar; sus simulaciones no demuestran rendimiento, exactitud deportiva ni portabilidad universal.


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:fps_nativo
# HUECO RESERVADO: extraccion a CADENCIA NATIVA. El mando no esta implementado a proposito.
print(PF.FPS_NATIVO_PENDIENTE)
# FPS_NATIVO = False   # <- aqui va el mando cuando lleguen las cifras de coste en tamano y tiempo


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:verificar_respaldo
# EJECUTAR UNA VEZ, con el token puesto. Crea el repo PRIVADO, sube un lote, lo lee, lo borra y
# confirma. Es lo que convierte "implementado" en "probado" sin que la credencial pase por otro
# sitio. El empaquetado, el troceado, el estado y la reanudacion YA estan probados en local; lo que
# esta celda anade es la subida autenticada y la comprobacion de que el repo es privado.
print(PF.CELDA_DE_VERIFICACION)
